In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as patches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import json
import numpy as np
from IPython.display import display, HTML
import matplotlib.ticker as ticker

# ==================== Konfigurasi Path ====================
base_path = "saved_models_lstm"
gambar_path = os.path.join(base_path, "gambar")
os.makedirs(gambar_path, exist_ok=True)
sentiment_folder = "saved_sentiment_scraping"
os.makedirs(sentiment_folder, exist_ok=True)

# ==================== Fungsi untuk menampilkan tabel dengan 4 digit ====================
def display_table(df, title):
    styled_df = df.copy()
    float_cols = styled_df.select_dtypes(include=['float']).columns
    for col in float_cols:
        styled_df[col] = styled_df[col].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else x)
    display(HTML(f"<h3>{title}</h3>"))
    display(styled_df)
    print("\n")

# ==================== Load Data ====================
raw_data_path = os.path.join(base_path, "raw_stock_prices_aapl.csv")
sentiment_path = os.path.join(sentiment_folder, "sentiment_aapl.csv")
processed_features_path = os.path.join(base_path, "processed_features_aapl.csv")

# Load data yang sudah diproses sebagai sumber utama (processed_features)
data_daily = pd.read_csv(processed_features_path, parse_dates=['Date'], index_col='Date')

# Load data mentah hanya untuk referensi (raw prices)
data_raw = pd.read_csv(
    raw_data_path,
    skiprows=3,
    header=None,
    names=['Date', 'Close'],
    parse_dates=['Date']
)
data_raw.set_index('Date', inplace=True)

# Load data sentimen
sentiment_df = pd.read_csv(sentiment_path, parse_dates=['Date'])
sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date']).dt.normalize()

# ==================== B. Pratinjau Dataset Saham & Sentimen ====================
# Rentang tanggal contoh (termasuk weekend)
tanggal_saham = pd.date_range('2020-12-14', '2020-12-21', freq='D').normalize()

# Tabel 1: Data saham dari processed features (reindex ke rentang harian supaya konsisten)
preview_processed = data_daily.reindex(tanggal_saham)[['Close']].sort_index()
preview_processed.index.name = 'Date'
preview_processed.to_csv(os.path.join(gambar_path, "preview_saham_mentah.csv"))
display_table(preview_processed.reset_index(), "Tabel 1: Pratinjau Data Saham (Sudah Diproses)")

# Tabel 2: Data sentimen (filter sesuai rentang)
sentiment_preview = sentiment_df[sentiment_df['Date'].isin(tanggal_saham)].sort_values('Date').copy()
sentiment_preview.to_csv(os.path.join(gambar_path, "preview_sentimen.csv"), index=False)
display_table(sentiment_preview, "Tabel 2: Pratinjau Data Sentimen")

# Tabel 3: Gabungan data (merge processed + sentiment)
left = preview_processed.reset_index()
right = sentiment_preview.copy()

# Pastikan kolom 'Date' ada di kedua df
if 'Date' not in left.columns:
    left.rename(columns={left.columns[0]: 'Date'}, inplace=True)
if 'Date' not in right.columns:
    right.reset_index(inplace=True)
    right.rename(columns={right.columns[0]: 'Date'}, inplace=True)

combined_preview = pd.merge(left, right, on='Date', how='left')
if 'Sentiment' not in combined_preview.columns:
    combined_preview['Sentiment'] = 0
combined_preview['Sentiment'] = combined_preview['Sentiment'].fillna(0)
display_table(combined_preview[['Date', 'Close', 'Sentiment']], "Tabel 3: Gabungan Data Saham dan Sentimen")

# ==================== C. Resampling & Overlay Sentimen (Perbaikan sesuai permintaan) ====================
# 1) Plot tanpa resampling/interpolasi: titik ORIGINAL dari raw_stock_prices_aapl.csv
# Ambil hanya baris raw yang berada di rentang tanggal contoh
raw_preview = data_raw[data_raw.index.normalize().isin(tanggal_saham)].sort_index()
# Normalisasi index ke tanggal tanpa jam
raw_preview.index = pd.to_datetime(raw_preview.index).normalize()

plt.figure(figsize=(12, 5))
plt.plot(raw_preview.index, raw_preview['Close'], 'o', color='tab:blue',
         label='Original', alpha=0.95,
         markeredgecolor='k', markersize=8)
plt.title("Tanpa Resampling & Interpolasi")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(gambar_path, "resample_no_interpolate.png"), dpi=300)
plt.close()

# 2) Plot dengan resampling/interpolasi: garis & titik harian dari processed_features_aapl.csv
# Gunakan preview_processed (reindexed ke harian). Interpolate bila ada NaN.
processed_series = preview_processed['Close'].copy()
processed_interp = processed_series.interpolate(method='time')  # isi NaN (weekend) berdasarkan waktu

plt.figure(figsize=(12, 5))
# garis interpolasi (dari processed_features)
plt.plot(processed_interp.index, processed_interp.values, '-', label='Resampled + Interpolated (processed)', zorder=1)
# titik harian interpolated berwarna orange sesuai legend
plt.plot(processed_interp.index, processed_interp.values, 's', markersize=6,
         label='Titik Harian (interpolated - processed)', color='orange', markeredgecolor='k', zorder=2)
# overlay titik original (dari raw) untuk perbandingan posisi — ditampilkan sebagai lingkaran berwarna biru lebih besar
plt.plot(raw_preview.index, raw_preview['Close'], 'o', color='navy',
         label='Original', markeredgecolor='k', markersize=8, zorder=3)

plt.title("Resampling & Interpolasi Harga Saham")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(gambar_path, "resample_with_interpolate.png"), dpi=300)
plt.close()

# 3) Overlay Harga (processed interpolated) & Sentimen (disesuaikan index)
# Siapkan series sentimen berdasarkan combined_preview (Date kolom)
sent_series = pd.Series(combined_preview['Sentiment'].values, index=pd.to_datetime(combined_preview['Date']).dt.normalize())
sent_for_plot = sent_series.reindex(processed_interp.index).fillna(0)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

# Harga: garis interpolated (processed) + titik harian orange (interpolated)
ax1.plot(processed_interp.index, processed_interp.values, 'b-', label='Harga')
ax1.plot(processed_interp.index, processed_interp.values, 's', markersize=6,
         color='orange', markeredgecolor='k', label='Titik Harian (interpolated - processed)', zorder=3)
# Titik asli raw (besar) untuk membandingkan
ax1.plot(raw_preview.index, raw_preview['Close'], 'o', color='navy', markersize=8,
         markeredgecolor='k', label='Original', zorder=4)

# Sentimen di axis kanan
ax2.plot(sent_for_plot.index, sent_for_plot.values, 'r--', label='Sentimen')

ax1.set_ylabel('Harga Saham (USD)', color='b')
ax2.set_ylabel('Sentimen Komposit', color='r')
ax1.tick_params(axis='y', labelcolor='b')
ax2.tick_params(axis='y', labelcolor='r')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.title("Overlay Harga Saham dan Sentimen")
plt.tight_layout()
plt.savefig(os.path.join(gambar_path, "overlay_sentimen_saham.png"), dpi=300)
plt.close()

# ==================== D. MA, RSI, Volatilitas (tetap) ====================
end_date = data_daily.index.max()
start_date = end_date - pd.DateOffset(days=100)
recent_data = data_daily.loc[start_date:end_date]

plt.figure(figsize=(12, 5))
plt.plot(recent_data.index, recent_data['Close'], label='Harga Saham')
plt.plot(recent_data.index, recent_data['MA_5'], label='MA 5 Hari')
plt.plot(recent_data.index, recent_data['MA_20'], label='MA 20 Hari')
plt.title("Moving Average 5 & 20 Hari")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(gambar_path, "ma5_ma20.png"), dpi=300)
plt.close()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax1.plot(recent_data.index, recent_data['RSI_14'], color='purple')
ax1.axhline(70, color='r', linestyle='--', alpha=0.3)
ax1.axhline(30, color='g', linestyle='--', alpha=0.3)
ax1.set_title('RSI 14 Hari')
ax1.set_ylim(0, 100)
ax1.grid(True)

ax2.plot(recent_data.index, recent_data['Volatility_5D'], color='orange')
ax2.set_title('Volatilitas 5 Hari')
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(gambar_path, "rsi_volatilitas.png"), dpi=300)
plt.close()

# ==================== Tabel 4: Preview Fitur Teknikal ====================
preview_fitur = data_daily.iloc[:5].copy()
preview_fitur = preview_fitur[[
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 
    'MA_5', 'MA_20', 'Return_1D', 'Volatility_5D', 
    'RSI_14', 'Momentum_5D', 'MA_5_20_ratio', 'RSI_MA5'
]]
preview_fitur.to_csv(os.path.join(gambar_path, "preview_fitur_teknikal.csv"))
display_table(preview_fitur.reset_index(), "Tabel 4: Pratinjau Fitur Teknikal")

# ==================== Tabel 5: Preview Normalisasi ====================
features = [
    'Sentiment_Lag1', 'MA_5', 'MA_20', 'Return_1D',
    'Volatility_5D', 'RSI_14', 'Momentum_5D',
    'MA_5_20_ratio', 'RSI_MA5', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5'
]
target = 'Close'

scaler = MinMaxScaler()
scaled = scaler.fit_transform(data_daily[features + [target]])
scaled_df = pd.DataFrame(scaled, columns=features + [target], index=data_daily.index)
scaled_df = scaled_df.iloc[:3]
scaled_df.to_csv(os.path.join(gambar_path, "preview_normalisasi.csv"))
display_table(scaled_df.reset_index(), "Tabel 5: Pratinjau Data Setelah Normalisasi")

# ==================== F. Gantt Chart Periode COVID ====================
covid_periods = {
    'Sebelum COVID': ('2017-04-09', '2020-02-29'),
    'Selama COVID':   ('2020-03-01', '2022-12-31'),
    'Setelah COVID':  ('2023-01-01', '2025-08-05')
}

rows = []
for p, (start, end) in covid_periods.items():
    data_p = data_daily.loc[start:end]
    split = int(len(data_p) * 0.8)
    rows.append({
        'Periode': p,
        'Jumlah Train': split,
        'Jumlah Test': len(data_p) - split,
        'Total': len(data_p)
    })
df_split = pd.DataFrame(rows)
df_split.to_csv(os.path.join(gambar_path, "pembagian_data_periode.csv"), index=False)
display_table(df_split, "Tabel 6: Pembagian Data per Periode COVID")

fig, ax = plt.subplots(figsize=(10, 3))
colors = ['green', 'orange', 'red']
for i, (label, (start, end)) in enumerate(covid_periods.items()):
    ax.barh(0, pd.to_datetime(end) - pd.to_datetime(start),
            left=pd.to_datetime(start), color=colors[i], label=label)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.yticks([])
plt.legend()
plt.title("Pembagian Periode COVID untuk Data")
plt.tight_layout()
plt.savefig(os.path.join(gambar_path, "gantt_covid_periods.png"), dpi=300)
plt.close()

# ==================== Tabel 7: Preview Data per Periode ====================
def find_examples(df, values):
    examples = []
    for val in values:
        idx = (df['Sentiment'] - val).abs().idxmin()
        examples.append(df.loc[[idx]])
    return pd.concat(examples)

covid_samples = []
for period, (start, end) in covid_periods.items():
    period_data = data_daily.loc[start:end]
    examples = find_examples(period_data, [0, 1, -1])
    examples['Periode'] = period
    covid_samples.append(examples)

covid_samples_df = pd.concat(covid_samples)
covid_samples_df = covid_samples_df[[
    'Close', 'Sentiment', 'Sentiment_Lag1', 'lag_1', 'lag_2', 'lag_3', 
    'lag_4', 'lag_5', 'MA_5', 'MA_20', 'Return_1D', 'Volatility_5D', 
    'RSI_14', 'Momentum_5D', 'MA_5_20_ratio', 'RSI_MA5', 'Periode'
]]
covid_samples_df.to_csv(os.path.join(gambar_path, "preview_data_periode.csv"))
display_table(covid_samples_df.reset_index(), "Tabel 7: Contoh Data per Periode COVID")

# ==================== Tabel 8: Preview Dataset Akhir ====================
preview_akhir = data_daily.reindex(tanggal_saham).head(5)
preview_akhir = preview_akhir[[
    'Close', 'Sentiment', 'Sentiment_Lag1', 'lag_1', 'lag_2', 'lag_3', 
    'lag_4', 'lag_5', 'MA_5', 'MA_20', 'Return_1D', 'Volatility_5D', 
    'RSI_14', 'Momentum_5D', 'MA_5_20_ratio', 'RSI_MA5'
]]
preview_akhir.to_csv(os.path.join(gambar_path, "preview_dataset_akhir.csv"))
display_table(preview_akhir.reset_index(), "Tabel 8: Pratinjau Dataset Akhir")

# ==================== G. Heatmap Korelasi ====================
plt.figure(figsize=(12, 10))
corr = data_daily[[
    'Close', 'Sentiment', 'Sentiment_Lag1', 'MA_5', 'MA_20', 
    'Return_1D', 'Volatility_5D', 'RSI_14', 'Momentum_5D',
    'MA_5_20_ratio', 'RSI_MA5', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5'
]].corr()
corr_display = corr.copy()
for col in corr_display.columns:
    corr_display[col] = corr_display[col].apply(lambda x: f"{x:.2f}")
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", 
            annot_kws={"size": 8}, cbar_kws={"shrink": 0.8})
plt.title("Heatmap Korelasi Antar Fitur", fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(gambar_path, "heatmap_korelasi.png"), dpi=300)
plt.close()

display_table(corr_display, "Tabel Korelasi Antar Fitur")

# ==================== H. Struktur Input LSTM ====================
fig, ax = plt.subplots(figsize=(8, 4))
rect = patches.Rectangle((0, 0), 9, 1, edgecolor='black', facecolor='skyblue')
ax.add_patch(rect)
ax.text(4.5, 0.5, '[samples, 1, features]', va='center', ha='center', fontsize=12)
plt.xlim(-1, 10)
plt.ylim(-0.5, 2)
plt.axis('off')
plt.title('Struktur Input ke LSTM')
plt.savefig(os.path.join(gambar_path, "lstm_input_shape.png"), dpi=300)
plt.close()

print("\n✅ Semua visualisasi dan pratinjau tabel berhasil disimpan di folder:", gambar_path)


,Date,Close
0,2020-12-14,118.6619
1,2020-12-15,124.6058
2,2020-12-16,124.5376
3,2020-12-17,125.4048
4,2020-12-18,123.4170
5,2020-12-19,123.4170
6,2020-12-20,123.4170
7,2020-12-21,124.9468


,Date,Sentiment
1345,2020-12-14,0.2880
1346,2020-12-15,0.2542
1347,2020-12-16,-0.2432
1348,2020-12-17,0.4168
1349,2020-12-18,0.2664
1350,2020-12-19,0.3825
1351,2020-12-20,0.3943
1352,2020-12-21,0.1653


,Date,Close,Sentiment
0,2020-12-14,118.6619,0.2880
1,2020-12-15,124.6058,0.2542
2,2020-12-16,124.5376,-0.2432
3,2020-12-17,125.4048,0.4168
4,2020-12-18,123.4170,0.2664
5,2020-12-19,123.4170,0.3825
6,2020-12-20,123.4170,0.3943
7,2020-12-21,124.9468,0.1653


,Date,lag_1,lag_2,lag_3,lag_4,lag_5,MA_5,MA_20,Return_1D,Volatility_5D,RSI_14,Momentum_5D,MA_5_20_ratio,RSI_MA5
0,2017-04-29,33.2850,33.3175,33.2920,33.4889,33.2827,33.3337,32.9803,0.0000,0.0043,68.0054,0.0023,1.0107,2.0401
1,2017-04-30,33.2850,33.2850,33.3175,33.2920,33.4889,33.2929,32.9859,0.0000,0.0027,68.0054,-0.2039,1.0093,2.0426
2,2017-05-01,33.2850,33.2850,33.2850,33.3175,33.2920,33.4273,33.0432,0.0204,0.0092,75.3467,0.6720,1.0116,2.2540
3,2017-05-02,33.9639,33.2850,33.2850,33.2850,33.3175,33.5997,33.1094,0.0063,0.0090,82.6260,0.8619,1.0148,2.4591
4,2017-05-03,34.1794,33.9639,33.2850,33.2850,33.2850,33.7577,33.1790,-0.0031,0.0094,83.2288,0.7901,1.0174,2.4655


,Date,Sentiment_Lag1,MA_5,MA_20,Return_1D,Volatility_5D,RSI_14,Momentum_5D,MA_5_20_ratio,RSI_MA5,lag_1,lag_2,lag_3,lag_4,lag_5,Close
0,2017-04-29,0.7134,0.0002,0.0000,0.4563,0.0372,0.6713,0.6111,0.6005,0.7686,0.0008,0.0010,0.0008,0.0017,0.0008,0.0008
1,2017-04-30,0.5569,0.0000,0.0000,0.4563,0.0204,0.6713,0.6081,0.5943,0.7696,0.0008,0.0008,0.0010,0.0008,0.0017,0.0008
2,2017-05-01,0.7480,0.0006,0.0003,0.5286,0.0871,0.7467,0.6208,0.6045,0.8501,0.0008,0.0008,0.0008,0.0010,0.0008,0.0038


,Periode,Jumlah Train,Jumlah Test,Total
0,Sebelum COVID,829,208,1037
1,Selama COVID,828,208,1036
2,Setelah COVID,758,190,948


,Date,Close,Sentiment,Sentiment_Lag1,lag_1,lag_2,lag_3,lag_4,lag_5,MA_5,MA_20,Return_1D,Volatility_5D,RSI_14,Momentum_5D,MA_5_20_ratio,RSI_MA5,Periode
0,2019-11-01,61.5990,0.0007,0.0576,59.8990,58.5747,58.5819,59.9688,59.3741,59.7247,58.1922,0.0284,0.0204,80.2430,2.2249,1.0263,1.3435,Sebelum COVID
1,2018-01-27,40.2052,0.8256,0.7848,40.2052,40.1114,40.8405,41.5015,41.4922,40.5727,41.2452,0.0000,0.0098,27.8565,-1.2870,0.9837,0.6866,Sebelum COVID
2,2019-09-15,52.6729,-0.2578,0.0701,52.6729,52.6729,53.7179,53.8383,52.1793,53.1150,51.2331,0.0000,0.0185,69.3916,0.4936,1.0367,1.3064,Sebelum COVID
3,2022-12-25,130.0262,0.0000,0.1439,130.0262,130.0262,130.3910,133.5663,130.4601,130.8072,135.7904,0.0000,0.0169,27.8209,-0.4339,0.9633,0.2127,Selama COVID
4,2021-02-14,132.1011,0.7845,0.1810,132.1011,132.1011,131.8669,132.1206,132.7256,132.0582,132.5428,0.0000,0.0024,67.8319,-0.6245,0.9963,0.5137,Selama COVID
5,2020-12-16,124.5376,-0.2432,0.2542,124.6058,118.6619,119.2758,119.2758,119.2758,121.2714,118.9815,-0.0005,0.0231,66.1433,5.2617,1.0192,0.5454,Selama COVID
6,2023-02-21,146.6386,0.0000,0.5263,150.6582,150.6582,150.6582,150.6582,151.8038,149.8543,150.5804,-0.0267,0.0116,32.1411,-5.1651,0.9952,0.2145,Setelah COVID
7,2025-05-25,195.0486,0.8519,0.3360,195.0486,195.0486,201.1317,201.8609,206.6255,197.6277,203.3924,0.0000,0.0142,45.3589,-11.5769,0.9717,0.2295,Setelah COVID
8,2025-06-28,200.8521,-0.0974,0.4271,200.8521,200.7721,201.3315,200.0729,201.2716,200.7761,199.1834,0.0000,0.0045,66.8855,-0.4195,1.0080,0.3331,Setelah COVID


,index,Close,Sentiment,Sentiment_Lag1,lag_1,lag_2,lag_3,lag_4,lag_5,MA_5,MA_20,Return_1D,Volatility_5D,RSI_14,Momentum_5D,MA_5_20_ratio,RSI_MA5
0,2020-12-14,118.6619,0.2880,0.3519,119.2758,119.2758,119.2758,120.0846,118.6619,119.3148,117.8303,-0.0051,0.0073,60.9113,0.0000,1.0126,0.5105
1,2020-12-15,124.6058,0.2542,0.2880,118.6619,119.2758,119.2758,119.2758,120.0846,120.2190,118.4076,0.0501,0.0239,67.2691,4.5212,1.0153,0.5596
2,2020-12-16,124.5376,-0.2432,0.2542,124.6058,118.6619,119.2758,119.2758,119.2758,121.2714,118.9815,-0.0005,0.0231,66.1433,5.2617,1.0192,0.5454
3,2020-12-17,125.4048,0.4168,-0.2432,124.5376,124.6058,118.6619,119.2758,119.2758,122.4972,119.5715,0.0070,0.0227,68.7014,6.1290,1.0245,0.5608
4,2020-12-18,123.4170,0.2664,0.4168,125.4048,124.5376,124.6058,118.6619,119.2758,123.3254,120.0622,-0.0159,0.0254,63.1643,4.1412,1.0272,0.5122


,Close,Sentiment,Sentiment_Lag1,MA_5,MA_20,Return_1D,Volatility_5D,RSI_14,Momentum_5D,MA_5_20_ratio,RSI_MA5,lag_1,lag_2,lag_3,lag_4,lag_5
Close,1.00,-0.35,-0.35,1.00,1.00,0.00,-0.06,-0.05,0.05,-0.00,-0.78,1.00,1.00,1.00,1.00,1.00
Sentiment,-0.35,1.00,0.76,-0.36,-0.36,0.06,-0.13,0.06,0.04,0.03,0.42,-0.35,-0.36,-0.36,-0.36,-0.36
Sentiment_Lag1,-0.35,0.76,1.00,-0.36,-0.36,-0.02,-0.15,0.06,0.03,0.03,0.42,-0.35,-0.36,-0.36,-0.36,-0.36
MA_5,1.00,-0.36,-0.36,1.00,1.00,-0.02,-0.06,-0.06,0.02,-0.01,-0.78,1.00,1.00,1.00,1.00,1.00
MA_20,1.00,-0.36,-0.36,1.00,1.00,-0.02,-0.04,-0.10,-0.01,-0.06,-0.79,1.00,1.00,1.00,1.00,1.00
Return_1D,0.00,0.06,-0.02,-0.02,-0.02,1.00,0.01,0.23,0.39,0.06,0.11,-0.03,-0.03,-0.03,-0.03,-0.03
Volatility_5D,-0.06,-0.13,-0.15,-0.06,-0.04,0.01,1.00,-0.20,-0.03,-0.28,-0.13,-0.06,-0.06,-0.05,-0.05,-0.05
RSI_14,-0.05,0.06,0.06,-0.06,-0.10,0.23,-0.20,1.00,0.46,0.84,0.49,-0.05,-0.06,-0.07,-0.07,-0.08
Momentum_5D,0.05,0.04,0.03,0.02,-0.01,0.39,-0.03,0.46,1.00,0.43,0.12,0.03,0.02,0.00,-0.01,-0.02
MA_5_20_ratio,-0.00,0.03,0.03,-0.01,-0.06,0.06,-0.28,0.84,0.43,1.00,0.38,-0.00,-0.01,-0.01,-0.02,-0.03





✅ Semua visualisasi dan pratinjau tabel berhasil disimpan di folder: saved_models_lstm\gambar


In [2]:
from PIL import Image

# Open original images
img1 = Image.open('saved_models_lstm/average_results/lstm_avg_plot_keseluruhan.png')
img2 = Image.open('saved_models_lstm/average_results/lstm_avg_plot_sebelum_covid.png')
img3 = Image.open('saved_models_lstm/average_results/lstm_avg_plot_selama_covid.png')
img4 = Image.open('saved_models_lstm/average_results/lstm_avg_plot_setelah_covid.png')

# Increase scale factor (2x or 3x)
scale_factor = 2
width, height = img1.size
target_size = (width * scale_factor, height * scale_factor)

# Resize with high-quality resampling (anti-aliasing)
img1 = img1.resize(target_size, Image.LANCZOS)
img2 = img2.resize(target_size, Image.LANCZOS)
img3 = img3.resize(target_size, Image.LANCZOS)
img4 = img4.resize(target_size, Image.LANCZOS)

# Create large canvas (2x2 grid)
combined_width = target_size[0] * 2
combined_height = target_size[1] * 2
combined_img = Image.new('RGB', (combined_width, combined_height), color=(255, 255, 255))

# Paste images
combined_img.paste(img1, (0, 0))
combined_img.paste(img2, (target_size[0], 0))
combined_img.paste(img3, (0, target_size[1]))
combined_img.paste(img4, (target_size[0], target_size[1]))

# Save as high-resolution PNG or PDF (for vector scaling)
combined_img.save('lstm_4plots_combined_highres.png', dpi=(300, 300))  # PNG 300 DPI
combined_img.save('lstm_4plots_combined.pdf')  # PDF untuk presentasi/skripsi
combined_img.show()